# Heinzy - shared Gemma 3 12B host (Google Colab)

**Issue:** [#16](https://github.com/MahikaGunjkar/AIGovernance/issues/16)

Runs **Ollama** with **gemma3:12b** on a Colab GPU and exposes it through **ngrok**.

Use **gemma3:12b** (Gemma 2 on Ollama is only 2B/9B/27B).

`
MODEL_ENDPOINT=https://<ngrok-host>
`

## Operator (@asriram15)

1. Runtime -> GPU (T4 or better).
2. Secrets -> NGROK_AUTHTOKEN with Notebook access.
3. Run cells in order.
4. Post MODEL_ENDPOINT=... to team chat on every restart.


In [ ]:
# 1) Confirm GPU
import subprocess
out = subprocess.getoutput('nvidia-smi')
print(out[:1200] if out.strip() else 'WARNING: no GPU')


In [ ]:
# 2) Install FULL Ollama into /usr/local (no symlink loops)
import os, json, shutil, subprocess, urllib.request, pathlib

os.environ['PATH'] = '/usr/local/bin:' + os.environ.get('PATH', '')
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
BIN = pathlib.Path('/usr/local/bin/ollama')

def is_elf(p: pathlib.Path) -> bool:
    try:
        # do not follow symlinks for the open check on broken loops
        if p.is_symlink():
            return False
        return p.is_file() and p.read_bytes()[:4] == b'\\x7fELF'
    except OSError:
        return False

subprocess.call(['pkill', '-f', 'ollama serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.check_call(['apt-get', 'update', '-qq'])
subprocess.check_call(['apt-get', 'install', '-y', '-qq', 'zstd', 'curl'])

# Break any existing symlink loop first
for p in [BIN, pathlib.Path('/usr/local/ollama')]:
    if p.is_symlink() or p.exists():
        try:
            p.unlink()
            print('removed', p)
        except IsADirectoryError:
            pass
        except Exception as e:
            print('could not remove', p, e)

with urllib.request.urlopen('https://api.github.com/repos/ollama/ollama/releases/latest', timeout=60) as r:
    release = json.load(r)
asset = next(a for a in release['assets'] if a['name'] == 'ollama-linux-amd64.tar.zst')
archive = pathlib.Path('/tmp/ollama-linux-amd64.tar.zst')
if not archive.exists() or archive.stat().st_size < 500_000_000:
    subprocess.check_call(['curl', '-L', '--fail', '--progress-bar', '-o', str(archive), asset['browser_download_url']])

print('Archive top entries:')
print(subprocess.getoutput(f'zstd -d -c {archive} | tar -tf - | head -40'))

print('Extracting to /usr/local ...')
subprocess.check_call(f'zstd -d -c {archive} | tar -xf - -C /usr/local', shell=True)

# Find REAL elf files named ollama (skip symlinks)
cands = []
for p in pathlib.Path('/usr/local').rglob('ollama'):
    try:
        if p.is_symlink():
            continue
        if is_elf(p):
            cands.append(p)
    except OSError:
        continue
print('real ollama ELF candidates:', cands)
servers = [p for p in pathlib.Path('/usr/local').rglob('llama-server') if not p.is_symlink()]
print('llama-server:', servers[:10])
assert cands, 'no real ollama ELF after extract'
assert servers, 'no llama-server after extract'

src = max(cands, key=lambda p: p.stat().st_size)
BIN.parent.mkdir(parents=True, exist_ok=True)
# COPY the binary into place — never symlink onto itself
if BIN.exists() or BIN.is_symlink():
    BIN.unlink()
shutil.copy2(src, BIN)
BIN.chmod(0o755)
print('installed', BIN, 'from', src, 'size', BIN.stat().st_size)
print(subprocess.getoutput('file /usr/local/bin/ollama'))
print(subprocess.getoutput('ollama --version'))
print(subprocess.getoutput('ls -la /usr/local/lib/ollama | head'))


In [ ]:
# 3) Start Ollama serve
import os, subprocess, time, urllib.request, pathlib

LOG = pathlib.Path('/tmp/ollama-serve.log')
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
os.environ['PATH'] = '/usr/local/bin:' + os.environ.get('PATH', '')

def ollama_up():
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
        return True
    except Exception:
        return False

if not ollama_up():
    env = os.environ.copy()
    env['OLLAMA_HOST'] = '0.0.0.0:11434'
    with LOG.open('wb') as logf:
        subprocess.Popen(['ollama', 'serve'], env=env, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True)
    for i in range(60):
        if ollama_up():
            print(f'up after {i+1}s')
            break
        time.sleep(1)
    else:
        print(LOG.read_text(errors='replace')[-4000:])
        raise RuntimeError('ollama serve failed')
else:
    print('already running')
print(urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=5).read()[:200])


In [ ]:
# 4) Pull gemma3:12b (team shared model for issue #16)
import os, subprocess, json, urllib.request

os.environ['PATH'] = '/usr/local/bin:' + os.environ.get('PATH', '')
MODEL = os.environ.get('HEINZY_OLLAMA_MODEL', 'gemma3:12b')
print('Pulling', MODEL)
proc = subprocess.run(['ollama', 'pull', MODEL], capture_output=True, text=True)
print(proc.stdout)
print(proc.stderr)
if proc.returncode != 0:
    raise RuntimeError(f'pull failed: {(proc.stderr or proc.stdout)[-1500:]}')
names = [m.get('name') for m in json.load(urllib.request.urlopen('http://127.0.0.1:11434/api/tags')).get('models', [])]
print('models:', names)
assert any(MODEL in (n or '') for n in names)


In [ ]:
# 5) ngrok tunnel
import os, subprocess, sys
from google.colab import userdata
from pyngrok import ngrok

token = userdata.get('NGROK_AUTHTOKEN')
if not token:
    raise RuntimeError('Add Colab Secret NGROK_AUTHTOKEN with Notebook access')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok'])
ngrok.kill()
ngrok.set_auth_token(token.strip())
tunnel = ngrok.connect(addr='11434', proto='http')
model_endpoint = tunnel.public_url.rstrip('/')
if model_endpoint.startswith('http://'):
    model_endpoint = 'https://' + model_endpoint[len('http://'):]
print('MODEL_ENDPOINT=' + model_endpoint)


In [ ]:
# 6) Self-check (diagnose HTTP 500)
# 500 on /api/chat is almost always model load/OOM, not ngrok.
import json, urllib.request, urllib.error, os, subprocess

MODEL = os.environ.get('HEINZY_OLLAMA_MODEL', 'gemma3:12b')
assert 'model_endpoint' in globals(), 'Run ngrok cell first'

print('--- GPU ---')
print(subprocess.getoutput('nvidia-smi')[:800])
print('--- ollama log tail ---')
print(subprocess.getoutput('tail -n 40 /tmp/ollama-serve.log'))

def call(url, data=None, timeout=300, headers=None):
    h = {'Content-Type': 'application/json'}
    if headers:
        h.update(headers)
    body = None if data is None else json.dumps(data).encode()
    req = urllib.request.Request(url, data=body, headers=h, method='GET' if data is None else 'POST')
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status, json.load(r), None
    except urllib.error.HTTPError as e:
        err = e.read().decode('utf-8', errors='replace')
        return e.code, None, err

status, tags, err = call('http://127.0.0.1:11434/api/tags')
print('tags status', status, 'models', None if not tags else [m.get('name') for m in tags.get('models', [])])
if err:
    print('tags error body:', err)

status, chat, err = call(
    'http://127.0.0.1:11434/api/chat',
    data={
        'model': MODEL,
        'messages': [{'role': 'user', 'content': 'Say pong'}],
        'stream': False,
        'options': {'num_predict': 8},
    },
)
print('chat status', status)
if chat:
    print('chat:', (chat.get('message') or {}).get('content', '')[:300])
if err:
    print('chat error body:', err[:2000])
    print()
    print('If this is OOM / runner crash: pull a smaller model and retry:')
    print("  !ollama pull gemma2:9b")
    print("  Then: import os; os.environ['HEINZY_OLLAMA_MODEL']='gemma2:9b'")
    print('  Re-run this cell (and set config MODEL_TAG/gemma2:9b for teammates).')
    raise RuntimeError(f'/api/chat HTTP {status}: {err[:500]}')

print('READY MODEL_ENDPOINT=' + model_endpoint)
